In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

TRAIN_PATH = "Train.csv"
TEST_PATH = "Test.csv"
TARGET = "Door leaf position"

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

# ------------------------------------------------------------
# 2. Parse Datetime: 2023-7-5-0-0-0-20 -> Timestamp
# ------------------------------------------------------------
def parse_datetime(s):
    parts = str(s).split("-")
    if len(parts) != 7:
        return pd.NaT
    y, m, d, H, M, S, ms = map(int, parts)
    return pd.Timestamp(
        year=y, month=m, day=d,
        hour=H, minute=M, second=S,
        microsecond=ms * 1000
    )

for df in [train, test]:
    df["Datetime"] = df["Datetime"].apply(parse_datetime)
    df.sort_values("Datetime", inplace=True)
    df["elapsed"] = (df["Datetime"] - df["Datetime"].min()).dt.total_seconds()

# ------------------------------------------------------------
# 3. Feature engineering
# ------------------------------------------------------------
def add_features(df, target=TARGET):
    df = df.copy()

    # command / state features
    df["command"] = df["Close command"].astype(int) - df["Open command"].astype(int)
    df["moving"] = df["Door is opening"].astype(int) + df["Door is closing"].astype(int)
    df["door_state"] = (
        df["Door Opened"].astype(int) * 2
        + df["Door Locked"].astype(int) * 3
        + df["Door is opening"].astype(int) * 5
        + df["Door is closing"].astype(int) * 7
    )

    # rolling and diff features from important sensors
    sensor_cols = [
        "Motor current(mA)",
        "Motor Voltage(10mV)",
        "Motor electrodynamic force",
    ]

    for col in sensor_cols:
        for w in [3, 5, 10, 20]:
            df[f"{col}_roll_mean_{w}"] = df[col].rolling(w, min_periods=1).mean()
            df[f"{col}_roll_std_{w}"] = df[col].rolling(w, min_periods=1).std().fillna(0)
        df[f"{col}_diff"] = df[col].diff().fillna(0)

    # past target features. These help a lot but use previous positions only.
    df["pos_lag1"] = df[target].shift(1)
    df["pos_lag2"] = df[target].shift(2)
    df["pos_diff1"] = df[target].diff().fillna(0)

    df = df.bfill().fillna(0)
    return df

train_fe = add_features(train)
test_fe = add_features(test)

# ------------------------------------------------------------
# 4. Feature sets
# ------------------------------------------------------------
drop_cols = ["Datetime", TARGET]
all_features = [c for c in train_fe.columns if c not in drop_cols]

# Baseline: no past target features
features_base = [
    c for c in all_features
    if c not in ["pos_lag1", "pos_lag2", "pos_diff1"]
]

# Improved: include past target features
features_improved = all_features

X_train_base = train_fe[features_base]
y_train = train_fe[TARGET]
X_test_base = test_fe[features_base]
y_test = test_fe[TARGET]

X_train_imp = train_fe[features_improved]
X_test_imp = test_fe[features_improved]

# ------------------------------------------------------------
# 5. Metrics
# ------------------------------------------------------------
def report(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"\n{name}")
    print(f"MAE  : {mae:.3f}")
    print(f"RMSE : {rmse:.3f}")
    print(f"R2   : {r2:.5f}")

    for tol in [0, 1, 2, 5, 10]:
        acc = (np.abs(y_true - y_pred) <= tol).mean()
        print(f"Tolerance {tol:>2} accuracy: {acc:.4f}")

# ------------------------------------------------------------
# 6. Baseline model
# ------------------------------------------------------------
baseline = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

baseline.fit(X_train_base, y_train)
pred_test_base = baseline.predict(X_test_base)

report("Baseline TEST", y_test, pred_test_base)

# ------------------------------------------------------------
# 7. Improved model
# ------------------------------------------------------------
improved = HistGradientBoostingRegressor(
    max_iter=1000,
    learning_rate=0.02,
    max_leaf_nodes=63,
    l2_regularization=0.1,
    random_state=42
)

improved.fit(X_train_imp, y_train)
pred_test_imp = improved.predict(X_test_imp)

report("Improved TEST", y_test, pred_test_imp)

# ------------------------------------------------------------
# 8. Optional: save predictions
# ------------------------------------------------------------
out = test_fe[["Datetime", TARGET]].copy()
out["pred_baseline"] = pred_test_base
out["pred_improved"] = pred_test_imp
out.to_csv("Test_predictions.csv", index=False)

print("\nSaved predictions to Test_predictions.csv")


Baseline TEST
MAE  : 24.551
RMSE : 54.697
R2   : 0.95470
Tolerance  0 accuracy: 0.0000
Tolerance  1 accuracy: 0.1590
Tolerance  2 accuracy: 0.2752
Tolerance  5 accuracy: 0.4801
Tolerance 10 accuracy: 0.6293

Improved TEST
MAE  : 2.031
RMSE : 7.874
R2   : 0.99906
Tolerance  0 accuracy: 0.0000
Tolerance  1 accuracy: 0.5602
Tolerance  2 accuracy: 0.7924
Tolerance  5 accuracy: 0.9589
Tolerance 10 accuracy: 0.9824

Saved predictions to Test_predictions.csv
